# Importing Libraries and dataset from kaggle

In [ ]:
!pip install -q kaggle

In [ ]:
import kagglehub

# Download the blood cell cancer dataset
path = kagglehub.dataset_download("mohammadamireshraghi/blood-cell-cancer-all-4class")

print("Path to dataset files:", path)

In [ ]:
import os
import shutil
import random

print("Contents of path:", os.listdir(path))

base_data_dir = os.path.join(path, "Blood cell Cancer [ALL]")

print("Contents of base_data_dir:", os.listdir(base_data_dir))

# Change split_data_root to a writable directory like /content/
split_data_root = os.path.join("/content/", "split_blood_cell_data")
train_dir = os.path.join(split_data_root, "train")
test_dir = os.path.join(split_data_root, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

train_split_ratio = 0.8
random.seed(42)

class_names = [d for d in os.listdir(base_data_dir) if os.path.isdir(os.path.join(base_data_dir, d))]

print(f"Detected classes: {class_names}")

for class_name in class_names:
    class_path = os.path.join(base_data_dir, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(images)

    train_count = int(len(images) * train_split_ratio)
    train_images = images[:train_count]
    test_images = images[train_count:]

    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img_name in train_images:
        src = os.path.join(class_path, img_name)
        dst = os.path.join(train_dir, class_name, img_name)
        shutil.copy(src, dst)

    for img_name in test_images:
        src = os.path.join(class_path, img_name)
        dst = os.path.join(test_dir, class_name, img_name)
        shutil.copy(src, dst)

print("Data splitting complete.")
print("Train directory:", train_dir, "Content:", os.listdir(train_dir)[:10])
print("Test directory:", test_dir, "Content:", os.listdir(test_dir)[:10])

# Data Preprocessing

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range = 0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
)

# Load MobileNetV2 Base and add the custom head

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization


NUM_CLASSES = train_gen.num_classes

base_model = MobileNetV2( input_shape=(224,224) + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = layers.Input(shape=(224,224) + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

Blood_tl_model = models.Model(inputs, outputs)

In [ ]:
print(train_gen.class_indices)

# Compiling

In [ ]:
import tensorflow as tf
Blood_tl_model.compile( optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Training

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

callbacks = [ ReduceLROnPlateau(monitor='val_loss',factor=0.3,patience=2,min_lr=1e-6,verbose=1),
EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True,verbose=1)
]

In [ ]:
history = Blood_tl_model.fit(train_gen,validation_data=test_gen,epochs=16, callbacks=callbacks)

# Accuracy

In [ ]:
loss, accuracy = Blood_tl_model.evaluate(test_gen)
print(f"Test loss: {loss}")
print(f"Test accuracy: {accuracy}")

# Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_gen.reset()
test_gen.shuffle = False

predictions = Blood_tl_model.predict(test_gen)
y_pred = np.argmax(predictions, axis=1)

y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Blood Cancer Cells Classification')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# 6. Detailed Performance Report
print(classification_report(y_true, y_pred, target_names=class_names))

# Single Prediction

In [ ]:
import os

print(os.listdir('.'))

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

image_path = '/Cell.jpg'
img_width, img_height = 224, 224

test_image = image.load_img(image_path, target_size=(img_width, img_height))
test_image = image.img_to_array(test_image)

test_image = test_image / 255.0

test_image = np.expand_dims(test_image, axis=0)

result = Blood_tl_model.predict(test_image)

class_indices = train_gen.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

predicted_index = np.argmax(result[0])
prediction = idx_to_class[predicted_index]

print(f"Predicted Type: {prediction}")
print(f"Confidence Scores: {result[0]}")

# Save the model

In [ ]:
Blood_tl_model.save("Blood_Cancer_model.h5")